In [1]:
import pandas as pd
import numpy as np

industry = pd.read_csv("../outputs/cleaned_industry.csv")
syllabus = pd.read_csv("../data/syllabus_dataset.csv")

syllabus["Skill"] = syllabus["Skill"].str.lower().str.strip()
syllabus["Institute"] = syllabus["Institute"].str.lower().str.strip()

print("Industry shape:", industry.shape)
print("Syllabus institutes:", syllabus["Institute"].unique())
industry.head()

Industry shape: (712, 6)
Syllabus institutes: ['rvce' 'msrit' 'bit' 'pesit' 'dsce']


,Company,Year,Role,Skill,Skill_Type,Upcoming_Flag
0,Oracle,2021,Backend Developer,java,Traditional,0
1,Oracle,2021,Backend Developer,spring boot,Transitional,0
2,Oracle,2021,Backend Developer,sql,Transitional,0
3,Oracle,2021,Software Engineer,java,Traditional,0
4,Oracle,2021,Software Engineer,data structures,Transitional,0


In [2]:
freq = industry.groupby("Skill").size().reset_index(name="Freq")
freq.head()

,Skill,Freq
0,advanced sql,8
1,api gateway,1
2,api rate limiting,1
3,auto scaling,1
4,aws,29


In [3]:
from sklearn.linear_model import LinearRegression

def compute_slope(skill_df):
    yearly = skill_df.groupby("Year").size().reset_index(name="Count")
    if len(yearly) > 1:
        X = yearly["Year"].values.reshape(-1, 1)
        y = yearly["Count"].values
        model = LinearRegression().fit(X, y)
        return model.coef_[0]
    else:
        return 0

trend = (
    industry.groupby("Skill")
    .apply(compute_slope, include_groups=False)
    .reset_index(name="Trend_Slope")
)
trend.head()

,Skill,Trend_Slope
0,advanced sql,-4.000000
1,api gateway,0.000000
2,api rate limiting,0.000000
3,auto scaling,0.000000
4,aws,-1.571429


In [4]:
max_year = industry["Year"].max()
min_year = industry["Year"].min()

industry["Recency"] = (industry["Year"] - min_year) / (max_year - min_year)

recency = industry.groupby("Skill")["Recency"].mean().reset_index(name="Recency_Weight")

print("Recency_Weight range:", recency["Recency_Weight"].min().round(3), "to", recency["Recency_Weight"].max().round(3))
recency.head()

Recency_Weight range: 0.0 to 1.0


,Skill,Recency_Weight
0,advanced sql,0.812500
1,api gateway,0.250000
2,api rate limiting,1.000000
3,auto scaling,0.500000
4,aws,0.318966


In [ ]:
upcoming = industry.groupby("Skill")["Upcoming_Flag"].max().reset_index()
upcoming.head()

,Skill,Upcoming_Flag
0,advanced sql,1
1,api gateway,0
2,api rate limiting,0
3,auto scaling,0
4,aws,0


In [7]:
features = freq.merge(trend, on="Skill")
features = features.merge(recency, on="Skill")
features = features.merge(upcoming, on="Skill")

features.head()

,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag
0,advanced sql,8,-4.000000,0.812500,1
1,api gateway,1,0.000000,0.250000,0
2,api rate limiting,1,0.000000,1.000000,0
3,auto scaling,1,0.000000,0.500000,0
4,aws,29,-1.571429,0.318966,0


In [8]:
msrit_skills = syllabus[syllabus["Institute"] == "msrit"]["Skill"].unique()
features["Is_Taught_MSRIT"] = features["Skill"].isin(msrit_skills).astype(int)

print("Is_Taught_MSRIT distribution:")
print(features["Is_Taught_MSRIT"].value_counts())

Is_Taught_MSRIT distribution:
Is_Taught_MSRIT
0    80
1    15
Name: count, dtype: int64


In [9]:
other_institutes = syllabus[syllabus["Institute"] != "msrit"]

taught_elsewhere = (
    other_institutes.groupby("Skill")["Institute"]
    .nunique()
    .reset_index(name="Taught_Elsewhere_Count")
)

features = features.merge(taught_elsewhere, on="Skill", how="left")
features["Taught_Elsewhere_Count"] = features["Taught_Elsewhere_Count"].fillna(0).astype(int)

print("Taught_Elsewhere_Count distribution:")
print(features["Taught_Elsewhere_Count"].value_counts().sort_index())
features.head()

Taught_Elsewhere_Count distribution:
Taught_Elsewhere_Count
0    74
1     6
2     4
3     5
4     6
Name: count, dtype: int64


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Elsewhere_Count
0,advanced sql,8,-4.000000,0.812500,1,1,2
1,api gateway,1,0.000000,0.250000,0,0,0
2,api rate limiting,1,0.000000,1.000000,0,0,0
3,auto scaling,1,0.000000,0.500000,0,0,0
4,aws,29,-1.571429,0.318966,0,1,4


In [10]:
numeric_cols = ["Freq", "Trend_Slope", "Recency_Weight", "Upcoming_Flag", "Taught_Elsewhere_Count"]
corr = features[numeric_cols].corr().round(2)
print("Feature correlation matrix:")
print(corr)

Feature correlation matrix:
                        Freq  Trend_Slope  Recency_Weight  Upcoming_Flag  \
Freq                    1.00        -0.27           -0.08          -0.06   
Trend_Slope            -0.27         1.00           -0.10          -0.17   
Recency_Weight         -0.08        -0.10            1.00           0.58   
Upcoming_Flag          -0.06        -0.17            0.58           1.00   
Taught_Elsewhere_Count  0.56        -0.23            0.02          -0.02   

                        Taught_Elsewhere_Count  
Freq                                      0.56  
Trend_Slope                              -0.23  
Recency_Weight                            0.02  
Upcoming_Flag                            -0.02  
Taught_Elsewhere_Count                    1.00  


In [11]:
features.to_csv("../outputs/feature_table.csv", index=False)
print("Saved feature_table.csv with shape:", features.shape)
print("Columns:", list(features.columns))

Saved feature_table.csv with shape: (95, 7)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Elsewhere_Count']
